In [18]:
#******************************************************
#*
#* Name:         nb-07-duck-db-exploration
#*     
#* Design Phase:
#*     Author:   John Miner
#*     Date:     04-01-2026
#*     Purpose:  Explore duck db for data engineering.
#* 
#******************************************************/

In [1]:
%%capture

#
# 0 - get lakehouse abfs path
#

def get_lakehouse_abfs_path(key = "/default"):
    
    # get mount point
    try:
        path = list(filter(lambda x: x["mountPoint"] == key, notebookutils.fs.mounts()))[0]["source"]
    except:
        path = ""

    # return results
    return path

# get the path
lakehouse_path = get_lakehouse_abfs_path()

In [2]:
#
#  1 - My very first query
#

#  add libs
import duckdb
import pandas

# answer to universe
duckdb.sql("SELECT 42 AS HITCH_HIKERS_GUIDE").show()

┌────────────────────┐
│ HITCH_HIKERS_GUIDE │
│       int32        │
├────────────────────┤
│                 42 │
└────────────────────┘



In [3]:
#
#  2 - Read all the stock data (65 s)
#

#  add libs
import duckdb
import pandas

# file locations
path = '/lakehouse/default/Files/stocks/**/*.CSV'

# fetch data
pdf1 = duckdb.read_csv(path, ignore_errors=True).fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
#
#  3 - Show record counts
#

pdf1.count()

symbol      635817
date        635817
open        635817
high        635817
low         635817
close       635817
adjclose    635817
volume      635817
dtype: int64

In [5]:
#
#  4 - Show pandas data
#

import pandas as pd 

# remove bad data
pdf2 = pdf1[pdf1['volume'] != 0]

# fix column names
pdf2.columns = pdf2.columns.str.replace('symbol', 'st_symbol')
pdf2.columns = pdf2.columns.str.replace('date', 'st_date')
pdf2.columns = pdf2.columns.str.replace('open', 'st_open')
pdf2.columns = pdf2.columns.str.replace('high', 'st_high')
pdf2.columns = pdf2.columns.str.replace('low', 'st_low')
pdf2.columns = pdf2.columns.str.replace('adjclose', 'st_adjclose')
pdf2.columns = pdf2.columns.str.replace('close', 'st_close')
pdf2.columns = pdf2.columns.str.replace('volume', 'st_volume')

# fix date
pdf2['st_date'] = pdf2['st_date'].dt.strftime('%Y-%m-%d')

# remove index
pdf3 = pdf2.reset_index(drop=True)

# show the data
pdf3.head(5)

/tmp/ipykernel_119/3849278522.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pdf2['st_date'] = pdf2['st_date'].dt.strftime('%Y-%m-%d')


,st_symbol,st_date,st_open,st_high,st_low,st_close,st_adjst_close,st_volume
0,A,2013-12-31,41.09442,41.16595,40.82976,40.90844,39.39425,1316000
1,A,2013-12-30,40.80830,41.10157,40.71531,41.00143,39.48380,1576900
2,A,2013-12-27,40.93705,41.07296,40.85837,40.89413,39.38047,913000
3,A,2013-12-26,40.94421,41.28040,40.94421,41.10157,39.48931,998700
4,A,2013-12-24,41.13018,41.20887,40.89413,40.93705,39.33123,1090000


In [6]:
pdf3.dtypes

st_symbol          object
st_date            object
st_open           float64
st_high           float64
st_low            float64
st_close          float64
st_adjst_close    float64
st_volume           int64
dtype: object

In [7]:
#
# 5 - remove target table
#

# set variables
schema_name = "bronze"
table_name = "duckdb_stock_data"
table_path = f"{lakehouse_path}/Tables/{schema_name}/{table_name}"

# only fails on missing dir
try:
    notebookutils.fs.rm(table_path, True)
except:
    pass

In [8]:

#
#  6 - Save stock data as delta table (4s)
#

# add libs
from deltalake import write_deltalake

# over write 
write_deltalake(table_path, pdf3, mode='overwrite')


In [10]:
%%tsql -artifact lh_python_vs_spark -type Lakehouse
-- select * from sys.objects where type = 'u'
select top 5 * from bronze.duckdb_stock_data;

In [11]:
#
#  6 - Create in memory table from csv files
#

import duckdb

# Connect to DuckDB (in-memory database)
con = duckdb.connect(database=':memory:')

# file locations
path = '/lakehouse/default/Files/stocks/**/*.CSV'

stmt = f"""
create table stocks as 
select 
  symbol as st_symbol , 
  cast(date as varchar) as st_date , 
  open as st_open , 
  high as st_high , 
  low as st_low , 
  close as st_close , 
  adjclose as st_adjclose , 
  volume as st_volume 
from read_csv_auto('{path}');
"""

# Create a table from a CSV file
con.execute(stmt)

# Verify the data
con.table("stocks").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬──────────┬──────────┬──────────┬──────────┬─────────────┬───────────┐
│ st_symbol │  st_date   │ st_open  │ st_high  │  st_low  │ st_close │ st_adjclose │ st_volume │
│  varchar  │  varchar   │  double  │  double  │  double  │  double  │   double    │   int64   │
├───────────┼────────────┼──────────┼──────────┼──────────┼──────────┼─────────────┼───────────┤
│ A         │ 2013-12-31 │ 41.09442 │ 41.16595 │ 40.82976 │ 40.90844 │    39.39425 │   1316000 │
│ A         │ 2013-12-30 │  40.8083 │ 41.10157 │ 40.71531 │ 41.00143 │     39.4838 │   1576900 │
│ A         │ 2013-12-27 │ 40.93705 │ 41.07296 │ 40.85837 │ 40.89413 │    39.38047 │    913000 │
│ A         │ 2013-12-27 │      0.0 │      0.0 │      0.0 │      0.0 │         0.0 │         0 │
│ A         │ 2013-12-26 │ 40.94421 │  41.2804 │ 40.94421 │ 41.10157 │    39.48931 │    998700 │
│ A         │ 2013-12-24 │ 41.13018 │ 41.20887 │ 40.89413 │ 40.93705 │    39.33123 │   1090000 │
│ A         │ 2013-12-23 │ 41.

In [12]:
#
# 7 - remove target table
#

# set variables
schema_name = "bronze"
table_name = "duckdb_top10_stocks"
table_path = f"{lakehouse_path}/Tables/{schema_name}/{table_name}"

# only fails on missing dir
try:
    notebookutils.fs.rm(table_path, True)
except:
    pass

In [13]:

#
#  8 - Save data for top 10 companies in 2025
#

# please note tesla was not on the list until 2020

# add libs
from deltalake import write_deltalake

# make sql stmt
stmt = """
select * 
from stocks 
where st_volume != 0 and 
st_symbol in 
( 
  'NVDA',
  'AAPL',
  'MSFT',
  'AMZN',
  'GOOGL',
  'GOOG',
  'AVGO',
  'FB',	
  'TSLA',
  'BRK-B'
)
"""

# get dataframe
pdf4 = con.execute(stmt).fetchdf()

# remove index
pdf5 = pdf4.reset_index(drop=True)

# over write 
write_deltalake(table_path, pdf4, mode='overwrite')

In [14]:
%%tsql -artifact lh_python_vs_spark -type Lakehouse
select top 5 * from bronze.duckdb_top10_stocks;


ProgrammingError: ('ID - 012f606d-f525-4330-83e2-3e8a6db6b19e | traceid = a30e5ca1-a716-41c6-ac18-0c568de55643 42S02', "[42S02] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Invalid object name 'bronze.duckdb_top10_stocks'. (208) (SQLExecDirectW)")

In [15]:
#
#  8 - Can read delta into pandas
#

import duckdb

# Query the Delta table
results = duckdb.execute(f"SELECT * FROM delta_scan('{table_path}');").fetchdf()

# sub-set eq MSFT
display(results[results["st_symbol"] == 'MSFT'])

In [16]:
#
#  9 - read high/low temps
#

import duckdb

# Connect to DuckDB (in-memory database)
con = duckdb.connect(database=':memory:')

# table list
table_lst = ['high_temps', 'low_temps']

# make tables
for table_nm in table_lst:

    # drop table
    stmt = f"drop table if exists {table_nm};"
    con.execute(stmt)

    # create table
    stmt = f"""
    create table {table_nm} as 
    select 
      date as obs_date , 
      temp as obs_temp 
    from read_csv_auto('/lakehouse/default/Files/weather/{table_nm}.csv');
    """
    con.execute(stmt)

    # show data
    con.table(table_nm).show()



┌────────────┬──────────┐
│  obs_date  │ obs_temp │
│    date    │  int64   │
├────────────┼──────────┤
│ 2015-01-01 │       42 │
│ 2015-01-02 │       42 │
│ 2015-01-03 │       41 │
│ 2015-01-04 │       51 │
│ 2015-01-05 │       54 │
│ 2015-01-06 │       54 │
│ 2015-01-07 │       46 │
│ 2015-01-08 │       46 │
│ 2015-01-09 │       50 │
│ 2015-01-10 │       46 │
│     ·      │        · │
│     ·      │        · │
│     ·      │        · │
│ 2018-09-21 │       72 │
│ 2018-09-22 │       68 │
│ 2018-09-23 │       65 │
│ 2018-09-24 │       68 │
│ 2018-09-25 │       69 │
│ 2018-09-26 │       70 │
│ 2018-09-27 │       73 │
│ 2018-09-28 │       79 │
│ 2018-09-29 │       66 │
│ 2018-09-30 │       65 │
├────────────┴──────────┤
│ 1369 rows (20 shown)  │
└───────────────────────┘

┌────────────┬──────────┐
│  obs_date  │ obs_temp │
│    date    │  int64   │
├────────────┼──────────┤
│ 2015-01-01 │       26 │
│ 2015-01-02 │       32 │
│ 2015-01-03 │       35 │
│ 2015-01-04 │       38 │
│ 2015-01-0

In [17]:
#
# 10 - remove target table
#

# set variables
schema_name = "bronze"
table_name = "duckdb_weather_data"
table_path = f"{lakehouse_path}/Tables/{schema_name}/{table_name}"

# only fails on missing dir
try:
    notebookutils.fs.rm(table_path, True)
except:
    pass

In [18]:

#
#  11 - data engineering with duckdb
#

# local var
table_nm = 'duckdb_weather_data'

# drop table
stmt = f"drop table if exists {table_nm};"
con.execute(stmt)

# create results table
stmt = f"""
create table {table_nm} as 
select 
    l.obs_date, 
    l.obs_temp as obs_low_temp,
    h.obs_temp as obs_high_temp
from 
    low_temps as l
join
    high_temps as h
on
    l.obs_date = h.obs_date
"""
con.execute(stmt)


# get dataframe
stmt = f"select * from {table_nm};"
pdf7 = con.execute(stmt).fetchdf()

# remove index
pdf8 = pdf7.reset_index(drop=True)

# over write 
write_deltalake(table_path, pdf8, mode='overwrite')

In [40]:
%%tsql -artifact lh_python_vs_spark -type Lakehouse
-- select * from sys.objects where type = 'u'
select top 10 * from bronze.duckdb_weather_data;

ProgrammingError: ('ID - 665e38da-ec8b-4d40-aa2c-4606c8570675 | traceid = 8e23774b-8a74-4f2a-a80e-7e1c2260b31a 42S02', "[42S02] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Invalid object name 'bronze.duckdb_weather_data'. (208) (SQLExecDirectW)")

In [19]:
# must be adls path
table_path = lakehouse_path + "/Tables/bronze/duckdb_weather_data"
import duckdb
display(duckdb.sql(f"select * from delta_scan('{table_path}') limit 1000 ").df())